In [1]:
import sys
sys.path.insert(0, "/Users/jacoboromerodiaz/Projects/gpt-2")

In [2]:
from gpt2.train import load_checkpoint
from gpt2.model import GPTConfig

import torch
device = 'cpu'

pretrained_checkpoint_file = "/Users/jacoboromerodiaz/Projects/gpt-2/gpt2/log/model_19072.pt"
sft_checkpoint_file = "/Users/jacoboromerodiaz/Projects/gpt-2/finetune/log/sft_model_05615.pt"
grpo_checkpoint_file = "/Users/jacoboromerodiaz/Projects/gpt-2/finetune/log/grpo_model_00205.pt"

pretrained_model, _ = load_checkpoint(pretrained_checkpoint_file, device, weights_only=False)
sft_model, _ = load_checkpoint(sft_checkpoint_file, device, weights_only=False)
grpo_model, _ = load_checkpoint(grpo_checkpoint_file, device, weights_only=False)

In [3]:
import tiktoken
from sft import extend_encoder
enc = tiktoken.get_encoding("gpt2")
finetune_enc = extend_encoder(enc)

In [4]:
def format_prompt(prompt):
    text = (
            f"<|im_start|>user\n{prompt}"
            + f"<|im_end|>\n<|im_start|>assistant\n"
        )
    return text

prompt = "How do I make an orange juice?"
finetune_prompt = format_prompt(prompt)

In [5]:
num_return_sequences = 1
max_length = 256

pretrained_model.eval()
tokens = enc.encode(prompt)
tokens = torch.tensor(tokens, dtype=torch.long)
tokens = tokens.unsqueeze(0).repeat(num_return_sequences, 1)  # (4, sequence_length)
x = tokens.to(device)

y = pretrained_model.generate(x, max_length, temperature=0.5, top_k=50)

for i in range(num_return_sequences):
        tokens = y[i, :].tolist()
        decoded = enc.decode(tokens)
        print(">", decoded)

> How do I make an orange juice?
Orange juice is an easy and easy way to make a delicious orange juice. It can be made with any fruit, and it is easy to make with just a few drops of lemon juice. Orange juice is generally sweet, and it is also a good source of vitamins C and A.
How do you make orange juice?
How to make orange juice:
- Wash the orange juice with warm water.
- Add 1/2 cup of water to the orange juice.
- Add a pinch of salt.
- Add a pinch of sugar.
- Add the juice to the water.
- Pour the juice into a glass.
- Pour the juice into the glass.
- When the juice is ready, add the lemon juice.
How do I make orange juice?
How to Make a Orange Juice:
- Place the orange juice in a saucepan, and add the water.
- Add the juice to the water.
- Add the lemon juice.
- Add the sugar.
- Add the orange juice.
- Add the orange juice.
- Pour the juice into the glass.
- Add the lemon juice, and the orange juice.
How do you make orange juice?
How to Make Orange Juice


In [6]:
import torch
from torch.nn import functional as F

IM_END = 50258
EOS = 50256

sft_model.eval()
grpo_model.eval()

tokens = finetune_enc.encode(finetune_prompt, allowed_special={"<|im_start|>", "<|im_end|>"})
tokens = torch.tensor(tokens, dtype=torch.long)
tokens = tokens.unsqueeze(0).repeat(num_return_sequences, 1)
x = tokens.to(device)

y_sft = sft_model.generate(x, max_length, temperature=0.5, top_k=50, stop_tokens=(IM_END, EOS))

for i in range(num_return_sequences):
    tokens = y_sft[i, :].tolist()
    decoded = finetune_enc.decode(tokens)
    print(">", decoded)

> <|im_start|>user
How do I make an orange juice?<|im_end|>
<|im_start|>assistant
Making an orange juice is easy. You can make your own orange juice by using a blender and making it into a smoothie. You can also add some fruit, such as apples or bananas, to make it easier to consume. You can also add some water, lemon juice or honey to make it more appealing.<|im_end|>


In [7]:
y_grpo = grpo_model.generate(x, max_length, temperature=0.5, top_k=50, stop_tokens=(IM_END, EOS))

for i in range(num_return_sequences):
    tokens = y_grpo[i, :].tolist()
    decoded = finetune_enc.decode(tokens)
    print(">", decoded)

> <|im_start|>user
How do I make an orange juice?<|im_end|>
<|im_start|>assistant
To make an orange juice, you need to add the following ingredients: 
- 1 tablespoon of orange juice
- 1 teaspoon of sugar
- 1 teaspoon of lemon juice
- 1/2 cup of water 
- 1/2 cup of sugar 
- 1/2 cup of artificial colors 
- 1/2 cup of unsweetened fruit juice 

Instructions:
1. Start by adding 1 tablespoon of orange juice to 1 cup of water. Add 1 teaspoon of sugar to 2 cups of water and mix until well combined.

2. Add 1/4 cup of water to the 2 cups of water and mix until well combined.

3. Add 1/2 cup of artificial colors to the water and mix until well combined.

4. Add 1/4 cup of water to the water and mix until well combined.

5. Pour the remaining 1/4 cup of water into a large bowl and add the orange juice.

6. Pour the orange juice into the bowl and mix until well combined.

7. Pour the remaining 1/4 cup of water into the bowl and add the sugar.

8. Let the orange juice sit for a few minutes and then